## Run evaluation on different action policies, e.g. VLA

In [ ]:
from VLABench.evaluation.evaluator import Evaluator
from VLABench.evaluation.model.policy.openvla import OpenVLA
from VLABench.evaluation.model.policy.base import RandomPolicy
from VLABench.tasks import *
from VLABench.robots import *

demo_tasks = ["select_fruit"]
unseen = True
save_dir = "/home/shiduo/project/VLABench/logs"

model_ckpt = "/remote-home1/pjliu/openvla-7b"
lora_ckpt = "/remote-home1/pjliu/openvla/weights/select_fruit+CSv1+lora/"

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"

### Init evaluator

In [ ]:
evaluator = Evaluator(
    tasks=demo_tasks,
    n_episodes=2,
    max_substeps=10,   
    save_dir=save_dir,
    visulization=True
)

### Load basic random policy

In [ ]:
random_policy = RandomPolicy(model=None)
result = evaluator.evaluate(random_policy)

### Load policies, take OpenVLA as example

In [ ]:
policy = OpenVLA(
    model_ckpt=model_ckpt,
    lora_ckpt=lora_ckpt,
    norm_config_file=os.path.join(os.getenv("VLABENCH_ROOT"), "configs/model/openvla_config.json")
)

result = evaluator.evaluate(policy)

## Run evaluation on different VLMs

In [1]:
from VLABench.evaluation.model.vlm import *
from VLABench.evaluation.evaluator import VLMEvaluator

import os
os.environ['DISPLAY']=':1'
vlm_name = "Qwen2_VL" # valid names: ["GPT_4v", "Qwen2_VL", "InternVL2", "MiniCPM_V2_6", "GLM4v", "Llava_NeXT"]
fewshot_num = 1
task_list = ["mesh_and_texture/select_fruit"]

def initialize_model(model_name, *args, **kwargs):
    cls = globals().get(model_name)
    if cls is None:
        raise ValueError(f"Model '{model_name}' not found in the current namespace.")
    
    return cls(*args, **kwargs)


/usr/local/lib/python3.10/dist-packages/glfw/__init__.py:917: GLFWError: (65550) b'X11: The DISPLAY environment variable is missing'
  warnings.warn(message, GLFWError)
/usr/local/lib/python3.10/dist-packages/dash/_jupyter.py:29: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  _dash_comm = Comm(target_name="dash")


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# vlm = initialize_model(vlm_name)
evaluator = VLMEvaluator(
    tasks=task_list,
    n_episodes=2,
    data_path="/workspace/robotics/dataset/vlabench/vlm_evaluation_v1.0/M&T",
    save_path=os.path.join(os.getenv("VLABENCH_ROOT"), "../logs/vlm"),
)

# evaluator.evaluate(vlm, few_shot_num=fewshot_num)
result=evaluator.get_final_score_dict(vlm_name, few_shot_num=1, with_CoT=False)


Load the task episodes by seeds, instead of episodes
/workspace/robotics/home_wzr/benchmark/VLABench/VLABench/../logs/vlm/Qwen2_VL/en/1_shot/final_score.json


In [8]:
import json

result_path = "/workspace/robotics/home_wzr/benchmark/VLABench/logs/vlm/Qwen2_VL/en/1_shot/final_score.json"
result = json.load(open(result_path, "r"))

all_skill_match_score = 0
all_entity_match_score = 0
all_skill_with_entity_match_score = 0
all_exact_match_score = 0
all_total_score = 0

task_num = 0

for task, sub_tasks in result.items():
    skill_match_score = 0
    entity_match_score = 0
    skill_with_entity_match_score = 0
    exact_match_score = 0
    total_score = 0
    for sub_task, info in sub_tasks.items():
        skill_match_score += info["skill_match_score"]
        entity_match_score += info["entity_match_score"]
        skill_with_entity_match_score += info["skill_with_entity_match_score"]
        exact_match_score += info["exact_match_score"]
        total_score += info["total_score"]
    all_skill_match_score += skill_match_score
    all_entity_match_score += entity_match_score
    all_skill_with_entity_match_score += skill_with_entity_match_score
    all_exact_match_score += exact_match_score
    all_total_score += total_score
    task_num += len(sub_tasks)
    print(f"{task}:")
    print(f"skill_match_score: {skill_match_score / len(sub_tasks)}")
    print(f"entity_match_score: {entity_match_score / len(sub_tasks)}")
    print(f"skill_with_entity_match_score: {skill_with_entity_match_score / len(sub_tasks)}")
    print(f"exact_match_score: {exact_match_score / len(sub_tasks)}")
    print(f"total_score: {total_score / len(sub_tasks)}")
    print("-"*50)
    
print(f"Average skill_match_score: {all_skill_match_score / task_num}")
print(f"Average entity_match_score: {all_entity_match_score / task_num}")
print(f"Average skill_with_entity_match_score: {all_skill_with_entity_match_score / task_num}")
print(f"Average exact_match_score: {all_exact_match_score / task_num}")
print(f"Average total_score: {all_total_score / task_num}")
        

select_poker:
skill_match_score: 59.0
entity_match_score: 13.0
skill_with_entity_match_score: 15.5
exact_match_score: 4.0
total_score: 28.8
--------------------------------------------------
select_drink:
skill_match_score: 50.0
entity_match_score: 22.0
skill_with_entity_match_score: 45.0
exact_match_score: 9.0
total_score: 28.8
--------------------------------------------------
select_billiards:
skill_match_score: 56.5
entity_match_score: 53.0
skill_with_entity_match_score: 23.5
exact_match_score: 7.0
total_score: 43.8
--------------------------------------------------
select_book:
skill_match_score: 55.0
entity_match_score: 61.0
skill_with_entity_match_score: 35.5
exact_match_score: 7.0
total_score: 46.4
--------------------------------------------------
select_toy:
skill_match_score: 58.0
entity_match_score: 55.5
skill_with_entity_match_score: 24.5
exact_match_score: 7.0
total_score: 45.4
--------------------------------------------------
select_fruit:
skill_match_score: 63.5
entity